In [2]:
from dataAnalysis.DataAnalysis import DataAnalysis
import pandas as pd
from xgboost import XGBClassifier
from EnsembleFramework import Framework
import numpy as np
import torch
from dataAnalysis.Constants import FEATURES, LABEL_COLUMN_NAME
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc
from hyperopt import fmin, tpe, hp,STATUS_OK, SparkTrials, space_eval 
from hyperopt import hp
from tqdm.notebook import tqdm

In [3]:
data = pd.read_csv(r"extdata/sbcdata.csv", header=0)
data_analysis = DataAnalysis(data)

/home/dwalke/git/sbc_app/dataAnalysis/data/Filter.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data['Label'] = self.data['Diagnosis']
/home/dwalke/git/sbc_app/dataAnalysis/data/Filter.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  self.data['Label'] = self.data['Diagnosis']
/home/dwalke/git/sbc_app/dataAnalysis/data/Filter.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the cave

In [4]:
sorted_train_data = data_analysis.get_training_data().sort_values(by="Id").reset_index(drop = True)
train_df = sorted_train_data.loc[:sorted_train_data.shape[0]*.8,:]
val_df = sorted_train_data.loc[sorted_train_data.shape[0]*.8:,:]
test_df = data_analysis.get_testing_data()
gw_df = data_analysis.get_gw_testing_data()

In [5]:
def convert_cat_feature(df):
    df = df.copy()
    df["SexCategory"] = (df["Sex"] == "W").astype(int)
    return df
    
def get_graph(df):
    edge_index = []
    df = convert_cat_feature(df)
    df = df.sort_values(by=["Id", "Time"]).reset_index(drop=True)
    
    ## group df by ids
    for identifier, group in df.groupby("Id"):
        offset = group.index[0]
        triu_matrix = np.triu((group.index.values + np.identity(1))[0])
        triu_exp_matrix = np.expand_dims(triu_matrix, axis=-1)
    
        idx_shape = group.index.shape[0]
        idx_matrix = np.ones((idx_shape, idx_shape)) * np.arange(idx_shape) + 1 + offset
        idx_matrix = np.transpose(idx_matrix)
        idx_exp_matrix = np.expand_dims(idx_matrix, axis=-1)
    
        unprocess_edges = np.concatenate((idx_exp_matrix, triu_exp_matrix), axis=-1)
        reshaped_unprocess_edges = np.reshape(unprocess_edges, (-1, 2))
        mask = (reshaped_unprocess_edges[:, 0] * reshaped_unprocess_edges[:, 1]) != 0
        edge_index.append((reshaped_unprocess_edges[mask] - 1).astype(np.int64))
    edge_index_torch = torch.from_numpy(np.concatenate(edge_index)).type(torch.long).transpose(0,1)
    features_torch = torch.from_numpy(df[FEATURES].values).type(torch.float)
    labels_torch = torch.from_numpy((df.sort_values(by=["Id", "Time"])[LABEL_COLUMN_NAME] == "Sepsis").values).type(torch.long)
    return features_torch, edge_index_torch, labels_torch

In [6]:
X_train_comp, edge_index_train_comp, y_train_comp = get_graph(sorted_train_data)
X_train, edge_index_train, y_train = get_graph(train_df)
X_val, edge_index_val, y_val = get_graph(val_df)
X_test, edge_index_test, y_test = get_graph(test_df)
X_gw, edge_index_gw, y_gw = get_graph(gw_df)

In [7]:
control_nodes = X_train_comp[y_train_comp == 0]
median_ref_node = torch.median(control_nodes, dim = 0)[0]
mean_ref_node = torch.mean(control_nodes, dim = 0)

In [8]:
def append_ref_node(X, edge_index, ref_node):
    X= torch.clone(X)
    X_new = torch.cat([X, ref_node.unsqueeze(0)], dim = 0)
    mask = torch.ones(X_new.shape[0], dtype= torch.bool)
    mask[-1] = False
    ref_target_nodes = torch.arange(X_new.shape[0])
    ref_source_nodes = torch.ones_like(ref_target_nodes) * (X_new.shape[0] -1)
    ref_edge_index = torch.stack([ref_source_nodes, ref_target_nodes])
    edge_index_new = torch.cat([edge_index, ref_edge_index], dim = -1)
    return X_new, edge_index_new, mask

In [9]:
rev_edge_index_train_comp = torch.zeros_like(edge_index_train_comp)
rev_edge_index_train_comp[0,:] = edge_index_train_comp[1,:]
rev_edge_index_train_comp[1,:] = edge_index_train_comp[0,:]

rev_edge_index_train = torch.zeros_like(edge_index_train)
rev_edge_index_train[0,:] = edge_index_train[1,:]
rev_edge_index_train[1,:] = edge_index_train[0,:]

rev_edge_index_val = torch.zeros_like(edge_index_val)
rev_edge_index_val[0,:] = edge_index_val[1,:]
rev_edge_index_val[1,:] = edge_index_val[0,:]

rev_edge_index_test = torch.zeros_like(edge_index_test)
rev_edge_index_test[0,:] = edge_index_test[1,:]
rev_edge_index_test[1,:] = edge_index_test[0,:]

rev_edge_index_gw = torch.zeros_like(edge_index_gw)
rev_edge_index_gw[0,:] = edge_index_gw[1,:]
rev_edge_index_gw[1,:] = edge_index_gw[0,:]

In [10]:
def get_features(framework, X, edge_index, mask):
    features = framework.get_features(X, edge_index, mask)
    return features

In [11]:
def get_sets_dict(ref_node, 
                     train_comp_edge_index,
                     train_edge_index,
                     val_edge_index,
                     test_edge_index,
                     gw_edge_index,
                    ):
    global X_train_comp, X_train, X_val, X_test, X_gw, y_train_comp, y_train, y_val, y_test, y_gw
    X_train_comp_new, edge_index_train_comp, mask_train_comp = append_ref_node(X_train_comp, train_comp_edge_index, ref_node)
    X_train_new, edge_index_train, mask_train = append_ref_node(X_train, train_edge_index, ref_node)
    X_val_new, edge_index_val, mask_val = append_ref_node(X_val, val_edge_index, ref_node)
    X_test_new, edge_index_test, mask_test = append_ref_node(X_test, test_edge_index, ref_node)
    X_gw_new, edge_index_gw, mask_gw = append_ref_node(X_gw, gw_edge_index, ref_node)
    sets = [("train_comp", X_train_comp_new, edge_index_train_comp, mask_train_comp, y_train_comp), ("train", X_train_new, edge_index_train,mask_train, y_train),
                ("val", X_val_new, edge_index_val,mask_val, y_val), ("test", X_test_new, edge_index_test,mask_test, y_test),  ("gw", X_gw_new, edge_index_gw,mask_gw, y_gw)]
    sets_dict = {graph_set[0]: (graph_set[1:]) for graph_set in sets}
    return sets_dict

In [12]:
median_dir_set_dict = get_sets_dict(median_ref_node, edge_index_train_comp, edge_index_train, edge_index_val, edge_index_test, edge_index_gw)
median_rev_dir_set_dict = get_sets_dict(median_ref_node, rev_edge_index_train_comp, rev_edge_index_train, rev_edge_index_val, rev_edge_index_test, rev_edge_index_gw)

mean_dir_set_dict = get_sets_dict(mean_ref_node, edge_index_train_comp, edge_index_train, edge_index_val, edge_index_test, edge_index_gw)
mean_rev_dir_set_dict = get_sets_dict(mean_ref_node, rev_edge_index_train_comp, rev_edge_index_train, rev_edge_index_val, rev_edge_index_test, rev_edge_index_gw)

In [13]:
def diff_user_function(kwargs):
    return kwargs["original_features"] - kwargs["mean_neighbors"]

def div_user_function(kwargs):
    return kwargs["original_features"] / kwargs["mean_neighbors"]

user_functions = {
    "diff": diff_user_function,
    "div": div_user_function
}

In [14]:
def test_auroc_and_auprc(framework, clf, X, edge_index,mask, y):
    features = torch.cat(get_features(framework, X, edge_index, mask), dim = 1)
    pred_proba = clf.predict_proba(features.cpu())[:,1]
    auroc = roc_auc_score(y, pred_proba)

    precision, recall, thresholds = precision_recall_curve(y, pred_proba)
    auprc = auc(recall, precision)
    return auroc, auprc

In [15]:
class SparkTune():
    def __init__(self, clf,user_function,hops,attention_config, y_train, X_train, train_edge_index, train_mask,
                               y_val, X_val, val_edge_index,val_mask, parallelism=1):
        self.clf = clf
        self.user_function = user_function
        self.hops = hops
        self.attention_config = attention_config
        self.parallelism = parallelism
        self.y_train = y_train
        self.X_train= X_train
        self.train_edge_index = train_edge_index
        self.train_mask = train_mask

        self.y_val = y_val
        self.X_val= X_val
        self.val_edge_index = val_edge_index
        self.val_mask = val_mask

        
        
    def objective(self, params):
        framework = Framework(user_functions=[self.user_function for i in self.hops], 
                     hops_list=self.hops,
                     clfs=[None for i in self.hops],
                     gpu_idx=0,
                     handle_nan=0.0,
                    attention_configs=[self.attention_config for i in self.hops])
        features_train = torch.cat(get_features(framework, self.X_train, self.train_edge_index, self.train_mask), dim = 1)
        features_val = torch.cat(get_features(framework, self.X_val, self.val_edge_index, self.val_mask), dim = 1)
            
        if "n_estimators" in params:
            params["n_estimators"] = int(params["n_estimators"])
        if "max_depth" in params:
            params["max_depth"] = int(params["max_depth"])
        if "max_leaf_nodes" in params:
            params["max_leaf_nodes"] = int(params["max_leaf_nodes"])
        model = self.clf(**params)
        
        model.fit(features_train.cpu(), self.y_train)
        
        y_pred_proba = model.predict_proba(features_val.cpu())[:, 1]
        score = roc_auc_score(self.y_val, y_pred_proba)
        return {'loss': -score, 'status': STATUS_OK}
    
    def search(self, space, max_evals):
        spark_trials = SparkTrials(parallelism = self.parallelism)
        best_params = fmin(self.objective, space, algo=tpe.suggest, trials=spark_trials, max_evals=max_evals, verbose = True)
        return space_eval(space, best_params)

In [16]:
control_weight = y_train.sum() / y_train.numel()
control_weight_scaled = (y_train.sum()*3) / (y_train.numel()*2)
xgb_choices = {
    "booster": ["gbtree", "dart"],
    "objective": ["binary:logistic"],
    "random_state": [42],
    "n_jobs": [-1],
}

xgb_space = {
    **{key: hp.choice(key, value) for key, value in xgb_choices.items()},
    'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.3)),
    'min_child_weight': hp.uniform('min_child_weight', 1, 10),
    'max_depth': hp.quniform('max_depth', 3, 15, 1),
    'subsample': hp.uniform('subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0),
    'n_estimators': hp.qloguniform('n_estimators', low=np.log(100), high=np.log(1000), q=50),
    'gamma': hp.uniform('gamma', 0, 5),
    'scale_pos_weight': hp.choice('scale_pos_weight', [1, control_weight.item(), control_weight_scaled.item()]),
    'lambda': hp.loguniform('lambda', np.log(1e-3), np.log(10)),
    'alpha': hp.loguniform('alpha', np.log(1e-3), np.log(10)),
}

In [17]:
edge_type_sets = {
    "dir_mean": mean_dir_set_dict,
    "dir_median": median_dir_set_dict,
    "rev_dir_mean": mean_rev_dir_set_dict,
    "rev_dir_median": median_rev_dir_set_dict,
}

In [ ]:
res_dict = dict()
for edge_type_set_name in tqdm(edge_type_sets):
    res_dict[edge_type_set_name] = dict()
    
    for user_function_key in tqdm(user_functions):
        res_dict[edge_type_set_name][user_function_key] = dict()
        
        user_function = user_functions[user_function_key]        
        print("Find best hyperparams")
        X_train, edge_index_train,train_mask, y_train = edge_type_sets[edge_type_set_name]["train"]
        X_val, edge_index_val, val_mask, y_val = edge_type_sets[edge_type_set_name]["val"]
        spark_tune = SparkTune(XGBClassifier,user_function,[0,1],None, y_train, X_train, edge_index_train, train_mask,
                               y_val, X_val, edge_index_val,val_mask, 4)
        params = spark_tune.search(xgb_space,100)
        if "n_estimators" in params:
            params["n_estimators"] = int(params["n_estimators"])
        if "max_depth" in params:
            params["max_depth"] = int(params["max_depth"])
        if "max_leaf_nodes" in params:
            params["max_leaf_nodes"] = int(params["max_leaf_nodes"])
        
        framework = Framework(user_functions=[user_function,user_function], 
                         hops_list=[0, 1],
                         clfs=[_, _],
                         gpu_idx=0,
                         handle_nan=0.0,
                        attention_configs=[None, None])
        print("Retrain with best params")
        X_train_comp, edge_index_train_comp,mask, y_train_comp = edge_type_sets[edge_type_set_name]["train_comp"]
        features_train = torch.cat(get_features(framework, X_train_comp, edge_index_train_comp, mask), dim = 1)
        model = XGBClassifier(**params)
        model.fit(features_train.cpu(), y_train_comp)
    
        print("Evaluate")
        eval_dict = dict()
        for name in edge_type_sets[edge_type_set_name]:
            X, edge_index,mask, y = edge_type_sets[edge_type_set_name][name]
            auroc, auprc = test_auroc_and_auprc(framework,model, X, edge_index,mask, y)
            eval_dict[name] = dict()
            eval_dict[name]["AUROC"] = np.round(auroc, 4)
            eval_dict[name]["AUPRC"] = np.round(auprc, 4)
        
        res_dict[edge_type_set_name][user_function_key]["best_params"] = params
        res_dict[edge_type_set_name][user_function_key]["eval_dict"] = eval_dict

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

Find best hyperparams


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/11/28 14:32:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/11/28 14:32:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/11/28 14:32:05 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
24/11/28 14:32:05 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
24/11/28 14:32:05 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.




  0%|                                                                           | 0/100 [00:00<?, ?trial/s, best loss=?]

...



 29%|████████████▏                             | 29/100 [3:27:47<14:25:49, 731.68s/trial, best loss: -0.956262928654445]



 30%|████████████▌                             | 30/100 [3:30:45<10:59:56, 565.66s/trial, best loss: -0.956262928654445]



 31%|█████████████▎                             | 31/100 [3:30:57<7:39:31, 399.58s/trial, best loss: -0.956262928654445]



 32%|█████████████▊                             | 32/100 [3:39:13<8:05:33, 428.43s/trial, best loss: -0.956262928654445]



 33%|██████████████▏                            | 33/100 [3:40:26<5:59:23, 321.85s/trial, best loss: -0.956262928654445]

In [ ]:
import pandas as pd
for key in res_dict:
    for function_key in res_dict[key]:
        print(key + " - " + function_key)
        print(pd.DataFrame(res_dict[key][function_key]["eval_dict"]))
        print(pd.DataFrame(res_dict[key][function_key]["eval_dict"]))
        print(res_dict[key][function_key]["best_params"])